### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 4: Generative AI. Exercise 2**

**Hidden Markov models: inferring what we cannot see**

We are inside a lab with no windows, so we cannot see the weather. We only see if a colleague comes in with an umbrella or not.
The weather is a hidden state that follows a Markov chain; the umbrella is what we observe. From the umbrellas we infer the probability that it is raining.

**The maths**

The hidden weather follows the transition matrix $P$ of the previous exercise. The emission matrix $E$ gives the probability of each observation for each weather: $E_{ij}$ is the probability of seeing $j$ (no umbrella, umbrella) when the weather is $i$.

Our belief $b_t$ is the probability of each weather on day $t$, given all the umbrellas seen so far. Every day we update it in two steps: move it one day forward with $P$, then multiply by how likely today's observation $o_t$ is under each weather, and divide by the total so it adds up to 1:

$$b_t(j) \propto E_{j,\, o_t} \sum_{i} b_{t-1}(i) \, P_{ij}$$

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# Hidden states and observations
states = ['sunny', 'rainy']
observations = ['no umbrella', 'umbrella']

# Transition matrix: row = today, column = tomorrow
P = np.array([
    [0.8, 0.2],
    [0.4, 0.6]])

# Emission matrix: row = weather, column = what we see
E = np.array([
    [0.9, 0.1],   # sunny: 10% of the time the colleague still brings an umbrella
    [0.2, 0.8]])  # rainy: 80% of the time the colleague brings an umbrella
print("P:\n", P)
print("E:\n", E)

In [ ]:
# Simulate 30 days: the true weather (hidden) and the umbrellas (observed)
np.random.seed(2)
n_days = 30
weather = np.zeros(n_days, dtype = int)
umbrella = np.zeros(n_days, dtype = int)

weather[0] = np.random.choice(2, p = [0.5, 0.5])
umbrella[0] = np.random.choice(2, p = E[weather[0]])
for t in range(1, n_days):
    weather[t] = np.random.choice(2, p = P[weather[t - 1]])
    umbrella[t] = np.random.choice(2, p = E[weather[t]])

print("Hidden weather (1 = rainy):", weather)
print("Umbrella (1 = yes):        ", umbrella)

In [ ]:
# Infer the weather from the umbrellas (forward algorithm)
belief = np.array([0.5, 0.5])   # day 0: no idea
prob_rain = []

for t in range(n_days):
    if t > 0:
        belief = belief @ P              # 1. move one day forward
    belief = belief * E[:, umbrella[t]]  # 2. how likely is today's observation
    belief = belief / np.sum(belief)     #    normalise so it adds up to 1
    prob_rain.append(belief[1])

prob_rain = np.array(prob_rain)
print("Probability of rain:", np.round(prob_rain, 2))

In [ ]:
# Compare the inferred probability with the true (hidden) weather
days = np.arange(n_days)
plt.figure(figsize = (10, 5))
plt.step(days, weather, where = 'mid', color = 'blue', label = 'True weather (1 = rainy)')
plt.plot(days, prob_rain, color = 'red', marker = 'o', label = 'Inferred probability of rain')
plt.scatter(days[umbrella == 1], np.full(np.sum(umbrella), 1.1), color = 'black', marker = 'v', label = 'Umbrella seen')
plt.xlabel('day')
plt.ylabel('rain')
plt.title('Hidden Markov model: inferring the weather from umbrellas')
plt.legend(loc = 'lower right')
plt.show()

accuracy = np.mean((prob_rain > 0.5) == weather)
print(f"Days where the most likely weather is the true one: {accuracy:.0%}")

In [ ]:
# Check by brute force on the first 5 days
# List every possible weather sequence, compute its probability, and add up those that end in rain
from itertools import product

n = 5
total = 0
rain_last_day = 0
for sequence in product([0, 1], repeat = n):
    prob = 0.5 * E[sequence[0], umbrella[0]]
    for t in range(1, n):
        prob *= P[sequence[t - 1], sequence[t]] * E[sequence[t], umbrella[t]]
    total += prob
    if sequence[-1] == 1:
        rain_last_day += prob

print(f"Brute force, P(rain on day {n - 1}): {rain_last_day / total:.4f}")
print(f"Forward algorithm:             {prob_rain[n - 1]:.4f}")
print("Close: ", np.isclose(rain_last_day / total, prob_rain[n - 1]))